In [ ]:
import torch
import numpy as np


def ellipse_perimeter(a, b, pi, one, three_not, three_one, ten, four, c1=None, c2=None):
    h = ((a - b) ** 2) / ((a + b) ** 2)
    base = (
        pi
        * (a + b)
        * (one + (three_not * h) / (ten + torch.sqrt(torch.relu(four - three_one * h))))
    )
    if c1 is not None and c2 is not None:
        base = base + c1 * h**2 + c2 * h**3
    return base


def _gl_nodes_weights(n: int, dtype, device):
    nodes, weights = np.polynomial.legendre.leggauss(n)
    return (
        torch.tensor(nodes, dtype=dtype, device=device),
        torch.tensor(weights, dtype=dtype, device=device),
    )


def recursive_perimeter(a, b, n: int = 128):
    nodes, weights = _gl_nodes_weights(n, dtype=a.dtype, device=a.device)

    theta = (torch.pi / 4) * (nodes + 1)
    cos_t = torch.cos(theta)
    sin_t = torch.sin(theta)

    if a.dim() > 0:
        a = a.unsqueeze(-1)
        b = b.unsqueeze(-1)

    integrand = torch.sqrt(a**2 * cos_t**2 + b**2 * sin_t**2)
    return 4 * (torch.pi / 4) * (weights * integrand).sum(-1)

In [ ]:
def compute_loss(approximator, a, b):
    constants = tuple(approximator[0])
    true_p = recursive_perimeter(a, b)
    approx_p = ellipse_perimeter(a, b, *constants)
    return ((true_p - approx_p) ** 2).mean()


ellipse_approximator = torch.tensor([3, 1, 3, 3, 10, 4], dtype=torch.float64).unsqueeze(
    0
)
ellipse_approximator = torch.randn_like(ellipse_approximator) + 2

In [ ]:
sample_size = 2**15
epochs = 10

ratios = torch.exp(
    torch.linspace(
        0.0,
        torch.log(torch.tensor(500.0, dtype=torch.float64)),
        sample_size,
        dtype=torch.float64,
    )
)
ellipses = torch.stack(
    [torch.ones(sample_size, dtype=torch.float64), ratios], dim=1
).repeat(epochs, 1)

In [ ]:
from tqdm import tqdm
from torch.utils.data import DataLoader


def train(ea, s, batch_size=1024):
    ea = ea.detach().clone().double().requires_grad_(True)

    optimizer = torch.optim.Adam([ea], lr=2e-3)
    loader = DataLoader(s, batch_size=batch_size, shuffle=True)
    pbar = tqdm(loader, desc="Training")

    for t in pbar:
        optimizer.zero_grad()
        a, b = t[:, 0].to(ea.device), t[:, 1].to(ea.device)
        loss = compute_loss(ea, a, b)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(mse=loss.item())

    return ea


ellipse_approximator = train(ellipse_approximator.to("cuda"), ellipses)

In [ ]:
test_ellipses = torch.stack(
    [
        torch.ones(512, dtype=torch.float64),
        torch.exp(
            torch.linspace(
                torch.log(torch.tensor(1.5, dtype=torch.float64)),
                torch.log(torch.tensor(500.0, dtype=torch.float64)),
                512,
                dtype=torch.float64,
            )
        ),
    ],
    dim=1,
).to("cuda")
test_a = test_ellipses[:, 0]
test_b = test_ellipses[:, 1]

ls_params = torch.tensor(
    [torch.pi, 1.0, 3.0, 3.0, 10.0, 4.0],
    dtype=torch.float64,
    device="cuda",
    requires_grad=True,
)
ls_optimizer = torch.optim.LBFGS([ls_params])


def ls_step():
    ls_optimizer.zero_grad()
    loss = compute_loss(ls_params.unsqueeze(0), test_a, test_b)
    loss.backward()
    return loss


for _ in range(100):
    ls_optimizer.step(ls_step)

with torch.no_grad():
    ramanujan = (
        5
        - compute_loss(
            torch.tensor([torch.pi, 1.0, 3.0, 3.0, 10.0, 4.0], dtype=torch.float64)
            .to("cuda")
            .unsqueeze(0),
            test_a,
            test_b,
        ).item()
    )
    me = 5 - compute_loss(ellipse_approximator.detach(), test_a, test_b).item()
    ls = 5 - compute_loss(ls_params.detach().unsqueeze(0), test_a, test_b).item()

while abs(me - 5) > 1e-4:
    ellipse_approximator = train(ellipse_approximator, ellipses.to("cuda"))
    with torch.no_grad():
        me = 5 - compute_loss(ellipse_approximator.detach(), test_a, test_b).item()

print(ramanujan, "to", ls, "to", me)

In [ ]:
pi, one, three_zero, three_one, ten, four = ellipse_approximator.tolist()[0]
print(
    rf"""$$
\begin{{aligned}}
h &= \frac{{(a - b)^2}}{{(a + b)^2}} \\
P(a, b) &= {pi} (a + b)\left(
{one} +
\frac{{{three_zero}\,h}}{{{ten} + \sqrt{{{four} - {three_one}\,h}}}}
\right)
\end{{aligned}}
$$"""
)